# 03 - Delay Analysis (SQLite)

## Objective
Answer the five business questions with SQL and summarize operational implications.

## Connect to SQLite

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path('../data/processed/vbz_delays.db')
conn = sqlite3.connect(db_path)
db_path

In [ ]:
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)

## Q1. Which transport lines experience the most delays?

In [ ]:
q1 = """
SELECT linie, COUNT(*) AS total_trips,
       ROUND(AVG(departure_delay_sec), 2) AS avg_departure_delay_sec,
       ROUND(100.0 * AVG(is_delayed_departure), 2) AS pct_delayed_departures
FROM delays_clean
GROUP BY linie
HAVING COUNT(*) >= 100
ORDER BY avg_departure_delay_sec DESC
LIMIT 15;
"""
df_q1 = pd.read_sql_query(q1, conn)
df_q1

## Q2. At what times of day are delays most frequent?

In [ ]:
q2 = """
SELECT departure_hour, COUNT(*) AS total_trips,
       ROUND(AVG(departure_delay_sec), 2) AS avg_departure_delay_sec,
       ROUND(100.0 * AVG(is_delayed_departure), 2) AS pct_delayed_departures
FROM delays_clean
GROUP BY departure_hour
ORDER BY departure_hour;
"""
df_q2 = pd.read_sql_query(q2, conn)
df_q2

## Q3. Which stops have the highest delay rates?

In [ ]:
q3 = """
SELECT halt_kurz_von AS stop_code, COUNT(*) AS total_departures,
       ROUND(AVG(departure_delay_sec), 2) AS avg_departure_delay_sec,
       ROUND(100.0 * AVG(is_delayed_departure), 2) AS pct_delayed_departures
FROM delays_clean
GROUP BY halt_kurz_von
HAVING COUNT(*) >= 100
ORDER BY pct_delayed_departures DESC
LIMIT 20;
"""
df_q3 = pd.read_sql_query(q3, conn)
df_q3

## Q4. How does actual travel time differ from scheduled time?

In [ ]:
q4 = """
SELECT linie, COUNT(*) AS total_segments,
       ROUND(AVG(scheduled_travel_sec), 2) AS avg_scheduled_travel_sec,
       ROUND(AVG(actual_travel_sec), 2) AS avg_actual_travel_sec,
       ROUND(AVG(travel_time_diff_sec), 2) AS avg_travel_time_diff_sec
FROM delays_clean
GROUP BY linie
HAVING COUNT(*) >= 100
ORDER BY avg_travel_time_diff_sec DESC
LIMIT 15;
"""
df_q4 = pd.read_sql_query(q4, conn)
df_q4

## Q5. Are delays increasing or decreasing over time?

In [ ]:
q5 = """
SELECT DATE(betriebsdatum) AS service_date,
       COUNT(*) AS total_trips,
       ROUND(AVG(departure_delay_sec), 2) AS avg_departure_delay_sec
FROM delays_clean
GROUP BY DATE(betriebsdatum)
ORDER BY DATE(betriebsdatum);
"""
df_q5 = pd.read_sql_query(q5, conn)
df_q5

## Load Precomputed Analysis Outputs

In [ ]:
q1_file = pd.read_csv('../data/processed/q1_delay_by_line.csv')
q2_file = pd.read_csv('../data/processed/q2_delay_by_hour.csv')
q3_file = pd.read_csv('../data/processed/q3_delay_by_stop.csv')
q4_file = pd.read_csv('../data/processed/q4_travel_time_diff.csv')
q5_file = pd.read_csv('../data/processed/q5_delay_trend.csv')

q1_file.head()

## Executive Summary
- The highest average line-level departure delays were on lines `91` (64.74 sec), `163` (46.85 sec), and `309` (42.52 sec).
- Delay frequency peaked in low-volume night windows (`01:00-02:00`), while high-volume daytime stress appeared around `14:00-15:00` (about 60-61% delayed departures).
- Several stops showed very high delay rates; with a minimum 500 departures, top hotspots included `BWEI` (92.19%), `BDIE` (90.47%), and `ITFA` (89.04%).
- Actual segment travel times were consistently above schedule for specific lines, with the largest gaps on lines `307`, `309`, and `91` (around 11 sec extra per segment).
- Daily delay increased strongly on `2022-12-31` (28.68 sec) versus the prior six-day average (11.00 sec), suggesting year-end operating pressure.